In [1]:
import os
import psycopg2
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

import datetime, uuid
from datetime import datetime
import numpy as np
from decimal import Decimal

In [2]:
sentence_transformer_ef = SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2',
    device='cpu',
    normalize_embeddings=True
)

# Initialize the ChromaDB client
client = chromadb.Client(Settings(
    persist_directory='/Users/gblasd/Documents/SmartBnB/db/chroma_db'
))

client = chromadb.PersistentClient(path='/Users/gblasd/Documents/SmartBnB/db/chroma_db')

/Users/gblasd/Documents/SmartBnB/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# empty the collection if it already exists
if False:
    vector_store = client.get_collection(name="example_collection")
    client.delete_collection('example_collection')

In [7]:
# create a collection if it does not exist
vector_store = client.get_or_create_collection(
    name="smartbnb_vector_store", 
    embedding_function=sentence_transformer_ef
)

In [13]:
# https://docs.trychroma.com/docs/querying-collections/metadata-filtering
client.list_collections()

[]

In [14]:
# Create connection to the database and initialize it
def create_db_connection() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5433"),
        dbname=os.getenv("DB_NAME", "smartbnb"),
        user=os.getenv("DB_USER", "admin"),
        password=os.getenv("DB_PASSWORD", "admin")
    )
    return conn

def drop_connection(conn):
    conn.close()

conn = create_db_connection()

def _sanitize_metadata_value(v):

    # to contain only str, int, float, or bool
    if isinstance(v, (str, int, float, bool)):
        return v
    elif v is None:
        return None
    elif isinstance(v, (Decimal,)):
        return float(v)
    elif isinstance(v, (list, tuple)):
        return [_sanitize_metadata_value(i) for i in v]
    elif isinstance(v, dict):
        return {k: _sanitize_metadata_value(v) for k, v in v.items()}
    elif isinstance(v, (set, frozenset)):
        return [_sanitize_metadata_value(i) for i in v]
    elif isinstance(v, (np.ndarray,)):
        return v.tolist()
    elif isinstance(v, (datetime.datetime, datetime.date)):
        return v.isoformat()
    elif isinstance(v, uuid.UUID):
        return str(v)
    return v

# Query the database to get all records from the listings table
with conn.cursor() as cur:
    cur.execute("""select l.id, l.listing_url, l.name, l.description, l.neighborhood_overview, l.neighbourhood_cleansed,
       l.property_type, l.room_type, l.accommodates, l.bathrooms, l.bathrooms_text, l.bedrooms, 
	   l.beds, l.amenities, l.price, l.latitude, l.longitude, l.minimum_nights, l.maximum_nights, 
	   l.has_availability, l.review_scores_accuracy, l.review_scores_communication,
	   l.review_scores_cleanliness, l.review_scores_location, l.review_scores_value, 
	   l.review_scores_rating, l.reviews_per_month, l.instant_bookable,
	   l.calculated_host_listings_count, l.calculated_host_listings_count_entire_homes,
	   l.calculated_host_listings_count_private_rooms, l.calculated_host_listings_count_shared_rooms
  from public.listings l
 where l.has_availability is true""")
    records = cur.fetchall()
    for record in records:
        # Add the record to the vector database collection
        row = {}
        row["metadata"] = [
            {
                col.name: _sanitize_metadata_value(record[i]) 
                for i, col in enumerate(cur.description) 
                if col.name  in ["neighbourhood_cleansed","property_type","room_type", "bathrooms", "bathrooms_text", "bedrooms", "beds",
                                 "price", "latitude", "longitude", "minimum_nights", "maximum_nights", "has_availability", 
                                 "review_scores_accuracy", "amenities"]
            }
        ]

        # Convert amenities from string to list
        if "amenities" in row["metadata"][0]:
            amenities_str = row["metadata"][0]["amenities"]
            amenities_list = [amenity.strip() for amenity in amenities_str.split(",")]
            row["metadata"][0]["amenities"] = amenities_list

        vector_store.add(
            ids=str(record[0]),
            documents=str(record[3]),
            metadatas=row["metadata"]
        )

drop_connection(conn)

In [15]:
query_results = vector_store.query(
    query_texts=["apartment near of independence angel with roof garden with pet friendly"],
    n_results=10,
    where = {
                "$and": [
                    # {
                    #     "price": {
                    #         "$gte": 1000 
                    #     }
                    # }, 
                    {
                        "price": {
                            "$lte": 3000 
                        }
                    },
                    {
                        "property_type": "Entire serviced apartment"
                    },
                    {
                        "neighbourhood_cleansed": "Cuauhtémoc"
                    },
                    {
                        "$or": [
                            {
                                "amenities" : {
                                    "$contains": "Wine glasses"
                                }
                            },
                             {
                                "amenities" : {
                                    "$contains": 'Pets allowed'
                                }
                            },
                        ]
                    }
                ]
            }
)

# Supported Operators Quick Reference
# $eq / $ne: Equal to / Not equal to
# $gt / $gte: Greater than / Greater than or equal to
# $lt / $lte: Less than / Less than or equal to
# $in / $nin: Contained in array / Not contained in

In [16]:
for k, v in enumerate(query_results['metadatas']):
    for val in query_results['metadatas'][0]:
        print(val['neighbourhood_cleansed'], val['price'], val['amenities'])

Cuauhtémoc 2140.0 ['Blender', 'Backyard', 'Coffee maker', 'Board games', 'Wine glasses', 'Coffee', 'Patio or balcony', 'Dedicated workspace', 'Heating', 'Window guards', 'Smart lock', 'Ethernet connection', 'BBQ grill', 'Extra pillows and blankets', 'Kitchen', 'Safe', 'Stove', 'Cooking basics', 'Single level home', 'Shampoo', 'Sound system with Bluetooth and aux', 'Elevator', 'Bikes', 'Outdoor furniture', 'Trash compactor', 'Wifi', 'Pocket wifi', 'Dryer', 'Air conditioning', 'Outlet covers', 'Hot water kettle', 'Hangers', 'Hair dryer', 'Body soap', 'Stainless steel oven', 'Dishes and silverware', 'Dining table', 'Microwave', 'Iron', 'High chair', 'Laundromat nearby', 'Toaster', 'Private entrance', 'First aid kit', 'Children\\u2019s books and toys', 'Essentials', 'Hot water', 'Paid street parking off premises', 'Long term stays allowed', 'Clothing storage: closet', 'Freezer', 'Cleaning products', 'Books and reading material', 'Conditioner', 'TV', 'Refrigerator', 'Free parking on premise

In [17]:
import pandas as pd

In [27]:
df = pd.DataFrame(query_results['metadatas'][0])
df["ids"] = query_results['ids'][0]
df["document"] = query_results["documents"][0]

In [28]:
df

,maximum_nights,room_type,has_availability,latitude,bathrooms_text,amenities,price,minimum_nights,neighbourhood_cleansed,longitude,bathrooms,beds,property_type,review_scores_accuracy,bedrooms,ids,document
0,90.0,Entire home/apt,True,19.43,2 baths,"[Blender, Backyard, Coffee maker, Board games,...",2140.0,5.0,Cuauhtémoc,-99.17,2.0,2.0,Entire serviced apartment,4.87,2.0,48610000,"Enjoy our new, cozy apartment, a short walk fr..."
1,1125.0,Entire home/apt,True,19.43,1 bath,"[Shampoo, Carbon monoxide alarm, Conditioner, ...",1575.0,1.0,Cuauhtémoc,-99.17,1.0,4.0,Entire serviced apartment,4.80,2.0,53951007,Excellent for a vacation with family or friend...
2,1125.0,Entire home/apt,True,19.43,1 bath,"[Dining table, Trash compactor, Hot water, Fre...",1743.0,3.0,Cuauhtémoc,-99.17,1.0,1.0,Entire serviced apartment,-1.00,1.0,49193169,Condominium with only 8 exclusive apartments j...
3,1125.0,Entire home/apt,True,19.43,2 baths,"[Carbon monoxide alarm, Hammock, Dedicated wor...",2659.0,2.0,Cuauhtémoc,-99.16,2.0,2.0,Entire serviced apartment,4.89,2.0,50150857,"Located in the Cuauhtémoc neighborhood, on Cal..."
4,365.0,Entire home/apt,True,19.43,1 bath,"[Clothing storage, Blender, Coffee maker, Coff...",1263.0,1.0,Cuauhtémoc,-99.17,1.0,3.0,Entire serviced apartment,4.85,2.0,734172134391391948,Excellent for a vacation with family or friend...
5,6.0,Entire home/apt,True,19.43,2 baths,"[Carbon monoxide alarm, Dedicated workspace, H...",1022.0,2.0,Cuauhtémoc,-99.17,2.0,3.0,Entire serviced apartment,4.91,2.0,680956160204051032,A spacious apartment. Very comfortable and wit...
6,1125.0,Entire home/apt,True,19.41,1 bath,"[Shampoo, Carbon monoxide alarm, Bluetooth sou...",2400.0,1.0,Cuauhtémoc,-99.17,1.0,2.0,Entire serviced apartment,4.94,1.0,1185004426591028575,Beautiful apartment in a building with a façad...
7,1125.0,Entire home/apt,True,19.43,1 bath,"[Shampoo, Carbon monoxide alarm, Conditioner, ...",2586.0,2.0,Cuauhtémoc,-99.17,1.0,2.0,Entire serviced apartment,4.96,1.0,1192109419712026503,With its prime location near Reforma Avenue an...
8,365.0,Entire home/apt,True,19.43,2 baths,"[Shampoo, Carbon monoxide alarm, Dedicated wor...",1795.0,1.0,Cuauhtémoc,-99.17,2.0,2.0,Entire serviced apartment,4.84,2.0,42936860,Boutique apartments that have everything you n...
9,365.0,Entire home/apt,True,19.43,2 baths,"[Shampoo, Carbon monoxide alarm, Shared patio ...",2128.0,1.0,Cuauhtémoc,-99.17,2.0,4.0,Entire serviced apartment,4.83,2.0,48069893,Boutique apartments that have everything you n...
